# Module 0. Observing the Earth

In this module, we'll explore how astronomers study exoplanets using our knowledge of our home world, Earth! 

## Learning goals: 
- Gain a conceptual understanding of flux, proper motion, and planetary transits
- Compare and contrast the light curves of a star that lacks a transiting plant and a star that possesses a transiting planet
- Explore how the transit method of exoplanet discovery operates

First, let's install our dependencies. We'll need this for the rest of the modules as well. 

In [1]:
import setup as s
from setup import * 

Installing dependencies...
Processing /home/conda/feedstock_root/build_artifacts/appnope_1733332318622/work (from -r requirements.txt (line 1))
Done!


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/System/Volumes/Data/home/conda/feedstock_root/build_artifacts/appnope_1733332318622/work'



Could not import regions, which is required for some of the functionalities of this module.
APOGEE allStar found!


Now, let's imagine you're another intelligent life form on an alien world looking for signs of life. After staring at a particular star for many days, you begin to notice periodic dips in the star's <b>lightcurve</b> every 365 days (let's pretend a day on your planet is equivalent to one Earth day). You speculate that these dips are the result of a planet passing in front of the star as present the following model:

<figure>
  <img src="helpful_figures/alien_earth_model.gif" style="display: block; margin-left: auto; margin-right: auto; width: 50%;">
  <div align="center">
  <p>An animation of the dynamics of your orbit and the orbit of the planet you believe you're observing (Earth!). </a> 
  </p>
  </div>
</figure>

However, you're told that these periodic dips are simply a result of the <b>proper motion </b> of your target star. <span style = "color: red" > How can you prove that they're not? </span>

You decide to conduct a study of several other stars within a few parsecs of you to prove that this star is exhibiting odd behavior. You investigate 5 stars that lack the periodic dips at 365 days that your other target star possesses. However, you do notice that these stars <b>and</b> your original target star seem to oscillate very slightly in the sky. You term this oscillation <b>proper motion</b> and record it in arcseconds. Additionally, you record the distance each star is from you (in parsecs) and its luminosity (in erg/s).

With this data, let's see if we can reconstruct the <b>lightcurves</b> you would have seen.

In [2]:
# We'll start by modeling the orbit of your home world first. 

# Define the distance your planet is from your host star in parsecs 
R_orbit = 1# pc 
# Define a year on your planet in terms of Earth days 
T = 10 # days 

# Choose how many observations you make relative to the period of your orbit
num_obs = 5000
time = np.linspace(0, 100*T, num_obs)

In [3]:
@widgets.interact(d=(0.5, 10, 0.5), mu=(1E-5, 1, 1E-5), Lsun = (1E25, 1E35, 10))
def update(d=1, mu = 1E-5, Lsun = 1E33):
    fig, ax = plt.subplots(1,1, figsize=(8, 4), sharex = True)
    [l.remove() for l in ax.lines]

    # define the x and y coordinates of your planet throughout its orbit 
    x1 = np.sqrt(d**2/2)
    y1 = x1 
    x,y = s.orbit(R_orbit, T, time)

    # convert from proper motion to transverse velocity 
    v_t = 4.74 * d* mu

    # define the x and y coordinates of the target star 
    x_star,y_star = s.target_star(x1,y1,v_t, time)

    # find the distance between you and the target star 
    d =  s.distance(x,y,x_star,y_star) 

    # use that distance to determine the flux
    solar_flux = s.flux(d, Lsun)
    ax.plot(time, np.log10(solar_flux)/np.mean(np.log10(solar_flux)), color = 'k')
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Log(Flux) (arbitrary)")
    ax.set_ylim(0, 2)

interactive(children=(FloatSlider(value=1.0, description='d', max=10.0, min=0.5, step=0.5), FloatSlider(value=…

Now, let's compare this to the case where we have a transiting planet. We are able to detect a transiting planet because it blocks a certain amount of light. To construct an accurate lightcurve, we need to know: 
1) How much light is blocked during the transit 
2) How long the transit lasts 
3) When the transit occurs 

It turns out that these quantities not only characterize our lightcurve, but they also tell us a lot of information about the host planet. 

Let's start with the first quantity. 

It is common practice to define a quantity $D$ known as the <b>transit depth </b>. $D$ is equivalent to the change in flux during transit over the total flux. Through the following algebraic steps, we learn that transit depth is proportional to the planet's radius. 

Let's define, $$D = \frac{F_{star} - F_{transit}}{F_{star}}$$

Then, 
$$F_{star} = \frac{L_{star}}{4\pi R_s^2}$$

$F_{transit}$ will be the fraction of $L_{star}$ observed over the area of the star that remains uncovered during the planet's transit. We can calculate this area: 
$$A_{uncovered} = A_{star} - A_{planet}$$
$$= 4\pi R_{star}^2 - 4\pi R_{planet}^2$$
$$= 4\pi(R_{star}^2 - R_{planet}^2)$$

Plugging in for $F_{star}$ and $F_{transit}$...
$$\Delta F = F_{star} - F_{transit} = \frac{L_{star}}{4\pi R_{star}^2}-\frac{L_{star}}{4\pi(R_{star}^2 - R_{planet}^2)}$$
$$= \frac{L_{star}}{4\pi}(\frac{1}{R_{star}^2}) - \frac{1}{(R_{star}^2 - R_{planet}^2)}$$
$$= \frac{L_{star}}{4\pi}(\frac{1}{R_{star}^2}) - \frac{1}{(R_{star}^2 - R_{planet}^2)}$$
$$= \frac{L_{star}}{4\pi} \frac{(R_{star}^2 - R_{planet}^2)}{R_{star}^2(R_{star}^2 - R_{planet}^2)} - \frac{R_{star}^2}{R_{star}^2(R_{star}^2 - R_{planet}^2)}$$
$$= \frac{L_{star}}{4\pi} \frac{(R_{star}^2 - R_{planet}^2) - R_{star}^2}{R_{star}^2(R_{star}^2 - R_{planet}^2)}$$
$$=\frac{L_{star}}{4\pi} \frac{R_{planet}^2}{R_{star}^2(R_{star}^2 - R_{planet}^2)}$$

Now, 
$$D = \frac{\Delta F}{F_{star}}$$
$$D= \frac{\frac{L_{star}}{4\pi} \frac{R_{planet}^2}{R_{star}^2(R_{star}^2 - R_{planet}^2)}}{\frac{L_{star}}{4\pi R_{star}^2}} = \frac{R_{planet}^2}{(R_{star}^2 - R_{planet}^2)}$$
$$D R_{star}^2- D R_{planet}^2 = R_{planet}^2$$
$$D R_{star}^2 = R_{planet}^2(1+D)$$
$$R_{planet}^2 = \frac{D R_{star}^2}{1+D}$$
$$ R_{planet} = \sqrt{\frac{D R_{star}^2}{1+D}}$$

Looking at the second quantity, we can see that the transit duration will be correlated to the speed at which the planet orbits the star. Pretty simply, we can say that our transit duration $t$ is: $$t = \frac{v_{transit}}{D_{star}}$$

This is just a reordering of our most basic kinematic equation $v = dx/dt$

But, $v_{transit}$ seems like a pretty hard quantity to measure. 

Fortunately, we can look at our third quantity, the transit period and obtain $v_{transit}$ from there. 

Again, with some basic classical mechanics, we see that 
$T = \frac{2\pi}{\omega}$ where  $\omega = \frac{v}{r}$. 

You may now guess that the $v$ in $v/r$ is $v_{transit}$ and you'd be correct! Additionally, $r$ is the radius of the orbit. 

To recap, by simply looking at a lightcurve of a transiting exoplanet, we should be able to obtain the planet's radius, period, and orbital radius (among other quantities, but we'll focus on these for now!). 

<figure>
  <img src="helpful_figures/exoplanet_observables.png" style="display: block; margin-left: auto; margin-right: auto; width: 50%;">
  <div align="center">
  <p> A diagram of how the radius of a planet ("R_p") and the orbital radius ("R_orb") can be obtained by observing a transit </a> 
  </p>
  </div>
</figure>


Since we know the radius, orbital period, and orbital radius of Earth and the radius of the sun, let's use those quantities to determine how Earth would appear as a transiting planet! 

In [4]:
R_earth = 6.3E8 # cm 
P_earth = 365 # days 
Rorb_earth = 1.4E13 # cm 
R_sun = 7E10 #cm 

In [5]:
D_earth = R_earth**2/(R_sun**2-R_earth**2)
t_earth = (2*np.pi*Rorb_earth)/(P_earth*2*R_sun)

In [6]:
# Choose how many observations you make relative to the period of your orbit
num_obs = 5000

# Choose how long you want to observe (you should make this at least one Earth period!)
time = np.linspace(0, 700, num_obs)

In [7]:
def tophat_function(time, flux, D, width=t_earth, period = P_earth):
    # Returns 1.0 inside the window, 0.0 outside
    relative_t = time % period
    idx = np.where(relative_t < width)[0]
    transit_flux = np.copy(flux)
    transit_flux[idx] = transit_flux[idx] -D*transit_flux[idx]
    return transit_flux

In [8]:
@widgets.interact(d=(0.5, 10, 0.5), mu=(1E-5, 100, 1E-5), Lsun = (1E25, 1E35, 10))
def update(d=1, mu = 1E-5, Lsun = 1E33):
    fig, ax = plt.subplots(2,1, figsize=(8, 4), sharex = True)
    [l.remove() for l in ax[0].lines]

    # define the x and y coordinates of your planet throughout its orbit 
    x1 = np.sqrt(d**2/2)
    y1 = x1 
    x,y = s.orbit(R_orbit, T, time)

    # convert from proper motion to transverse velocity 
    v_t = 4.74 * d* mu

    # define the x and y coordinates of the target star 
    x_star,y_star = s.target_star(x1,y1,v_t, time)

    # find the distance between you and the target star 
    d =  s.distance(x,y,x_star,y_star) 

    # use that distance to determine the flux
    solar_flux = s.flux(d, Lsun)
    tophat= tophat_function(time, solar_flux, D_earth)
    ax[0].plot(time, np.log10(tophat)/np.log10(np.mean(solar_flux)), color = 'k')
    ax[0].axvline(P_earth, color = 'blue', linestyle = '--', alpha = 0.5)
    ax[0].set_xlabel("Time (days)")
    ax[0].set_ylabel("Log(Flux) (arbitrary)")
    ax[0].set_ylim(0, 2)
    ax[1].plot(time, (solar_flux-tophat)/np.mean(solar_flux), color = 'k')
    ax[1].axvline(P_earth, color = 'blue', linestyle = '--', alpha = 0.5)
    ax[1].set_ylim(-2E-3, 2E-3)
    ax[1].set_xlabel("Time (days)")
    ax[1].set_ylabel(r"$(F_s - F_t)/F_s$")

interactive(children=(FloatSlider(value=1.0, description='d', max=10.0, min=0.5, step=0.5), FloatSlider(value=…

You shouldn't really notice much of a change here from our earlier picture. In the top panel, we've plotted the logarithm of the flux and it looks identical to our earlier picture. In the bottom panel, we've taken the difference between the flux of our star with no transit present and the transit light curve and normalized this by the flux of our star. If you look <em>very</em> closely, you can make out a little bump at $t = 0$ and $t=365$. But, they're effectively indistinguishable. 

Let's investigate if our math is wrong or if Earth is just particularly difficult to detect by varying the parameters of our planetary target. 

In [ ]:
Lsun = 1E33 # set this to about the luminosity of the sun 
mu = 1E-5 # randomly chosen 

@widgets.interact(d=(0.5, 10, 0.5), R_pl = (0.5, 30, 0.5), R_orb=(0.5, 30, 0.5), P_pl = (0.5, np.max(time), 0.5), R_sol = (1E-2, 100, 0.5))
def update(d=1, R_pl=1, R_orb = 1, P_pl=1, R_sol = 1):
    fig, ax = plt.subplots(2,1, figsize=(8, 4), sharex = True)
    [l.remove() for l in ax[0].lines]
    [l.remove() for l in ax[1].lines]
    R_sol = R_sol * R_sun # convert from solar radii to cm 
    R_pl = R_pl * R_earth # convert from Earth radii to cm 
    R_orb = R_orb * Rorb_earth # convert from orbital radius of Earth to cm 
    #P_pl = P_pl * P_earth # convert from Earth period to days 
    tr_depth = R_pl**2/(R_sun**2-R_pl**2)
    tr_time = (2*np.pi*R_orb)/(P_pl*2*R_sun)
    # define the x and y coordinates of your planet throughout its orbit 
    x1 = np.sqrt(d**2/2)
    y1 = x1 
    x,y = s.orbit(R_orbit, T, time)

    # convert from proper motion to transverse velocity 
    v_t = 4.74 * d* mu

    # define the x and y coordinates of the target star 
    x_star,y_star = s.target_star(x1,y1,v_t, time)

    # find the distance between you and the target star 
    d =  s.distance(x,y,x_star,y_star) 

    # use that distance to determine the flux
    solar_flux = s.flux(d, Lsun)
    ax[0].plot(time, np.log10(solar_flux)/np.log10(np.mean(solar_flux)))
    tophat= tophat_function(time, solar_flux, tr_depth, width = tr_time, period = P_pl)
    print(f"transit depth: {tr_depth:.2f}")
    print(f"transit time: {tr_time:.2f} days")
    ax[1].plot(time, (solar_flux-tophat)/np.mean(solar_flux), color = 'k')
    ax[1].set_xlabel("Time (days)")
    ax[0].set_ylabel("Log(Flux) (arbitrary)")
    ax[1].set_ylabel(r"$(F_s - F_t)/F_s$")
    ax[1].set_ylim(0, 4)
    #x.set_ylim(0, 2)

interactive(children=(FloatSlider(value=1.0, description='d', max=10.0, min=0.5, step=0.5), FloatSlider(value=…

Congrats! You've successfully proved that there is <b>both</b> a variation in the signal due to the proper motion of the star and due to the transit of a planet! Visit <a href = "./MODULE1_proto.ipynb"> <code> MODULE1.ipynb </code> </a> to apply your knowledge to real exoplanet lightcurves. 